# MLP Multivariate Baseline

Runs `MLPMultivariateModel` on all five benchmark datasets in two modes:

* **Part 1 — Default config (no tuning):** applies only the shared dataset
  training schedule (batch size, epochs, LR milestones) and uses the
  architecture defaults (`hidden_size=512`, `num_layers=2`, `dropout=0.1`).
  Good for a quick sanity-check before committing to a full tuning run.

* **Part 2 — Hyperparameter tuning:** 18 random trials (seed 0) searching
  over `lr` (log-uniform), `dropout`, `hidden_size`, and `num_layers`,
  exactly matching the protocol used for GNN models in `examples_new/`.
  The best config found on the validation set is then evaluated once on
  the held-out test set for an unbiased final number.

In [ ]:
import sys
from dataclasses import replace
sys.path.insert(0, "..")

from gnn_benchmark.benchmark import BenchmarkRunner
from gnn_benchmark.models import MLPMultivariateModel, MLPMultivariateConfig
from gnn_benchmark.tuning import HyperparameterTuner
from gnn_benchmark.tuning.spaces import mlp_multivariate_search_space

WORKSPACE = "../benchmark_workspace"
N_TRIALS  = 18
SEED      = 0
LR_LOW    = 1e-4
LR_HIGH   = 5e-3

# Per-dataset training schedule — identical to the one used in examples_new/
# for fair comparison across models.
DATASET_SCHEDULE = {
    "noaa-buoy":              dict(batch_size=32, max_epochs=20, early_stop=7, lr_milestones=[10, 15], lr_decay_ratio=0.5),
    "eu-load":                dict(batch_size=32, max_epochs=20, early_stop=7, lr_milestones=[10, 15], lr_decay_ratio=0.5),
    "lamah-ce-dynamic":       dict(batch_size=16, max_epochs=20, early_stop=7, lr_milestones=[10, 15], lr_decay_ratio=0.5),
    "nyc-covid":              dict(batch_size=8,  max_epochs=20, early_stop=7, lr_milestones=[10, 15], lr_decay_ratio=0.5),
    "divvy-bikeshare-static": dict(batch_size=16, max_epochs=20, early_stop=7, lr_milestones=[10, 15], lr_decay_ratio=0.5),
}

def make_config(dataset_key):
    """Return MLPMultivariateConfig with the dataset's training schedule applied."""
    return replace(MLPMultivariateConfig(), **DATASET_SCHEDULE[dataset_key])


---
## Part 1 — Default Config (No Tuning)

Each cell below applies the dataset's memory-safe training schedule
(batch size, max epochs, early-stop patience, LR milestones) but leaves
all architecture knobs at their defaults.  This is the fastest way to
get a number out of the model before deciding whether to tune.

### NOAA Buoy

In [ ]:
DATASET = "noaa-buoy"
cfg = make_config(DATASET)
print(f"Config: {cfg}")

runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
result = runner.run(MLPMultivariateModel(), config=cfg)
print(result.summary())


### EU Load

In [ ]:
DATASET = "eu-load"
cfg = make_config(DATASET)
print(f"Config: {cfg}")

runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
result = runner.run(MLPMultivariateModel(), config=cfg)
print(result.summary())


### LamaH-CE Dynamic

In [ ]:
DATASET = "lamah-ce-dynamic"
cfg = make_config(DATASET)
print(f"Config: {cfg}")

runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
result = runner.run(MLPMultivariateModel(), config=cfg)
print(result.summary())


### NYC COVID

In [ ]:
DATASET = "nyc-covid"
cfg = make_config(DATASET)
print(f"Config: {cfg}")

runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
result = runner.run(MLPMultivariateModel(), config=cfg)
print(result.summary())


### Divvy Bikeshare

In [ ]:
DATASET = "divvy-bikeshare-static"
cfg = make_config(DATASET)
print(f"Config: {cfg}")

runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
result = runner.run(MLPMultivariateModel(), config=cfg)
print(result.summary())


---
## Part 2 — Hyperparameter Tuning (18 Random Trials)

Same training schedule as Part 1, but now the tuner samples 18 random
configurations from the MLP search space
(`lr`, `dropout`, `hidden_size`, `num_layers`) and selects the best one
by validation loss.  The winning config is then run once on the test set.

This follows the identical protocol used for all GNN models in
`examples_new/` so results are directly comparable.

### NOAA Buoy

In [ ]:
DATASET = "noaa-buoy"
base_cfg = make_config(DATASET)

tuner = HyperparameterTuner(
    model_factory=MLPMultivariateModel,
    base_config=base_cfg,
    dataset_key=DATASET,
    workspace_dir=WORKSPACE,
    search_space=mlp_multivariate_search_space(lr_low=LR_LOW, lr_high=LR_HIGH),
    strategy="random",
    n_trials=N_TRIALS,
    seed=SEED,
)
tuning_result = tuner.run()
print(tuning_result.summary())
print(
    f"Tuning compute: {tuning_result.total_compute_time_sec:.2f}s"
    f" across {len(tuning_result.trials)} trial(s)."
)

if tuning_result.best is not None:
    print("\nRunning final test-set evaluation with best config …")
    runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
    final = runner.run(MLPMultivariateModel(), config=tuning_result.best.config)
    print(final.summary())
else:
    print("No successful trials — skipping final evaluation.")


### EU Load

In [ ]:
DATASET = "eu-load"
base_cfg = make_config(DATASET)

tuner = HyperparameterTuner(
    model_factory=MLPMultivariateModel,
    base_config=base_cfg,
    dataset_key=DATASET,
    workspace_dir=WORKSPACE,
    search_space=mlp_multivariate_search_space(lr_low=LR_LOW, lr_high=LR_HIGH),
    strategy="random",
    n_trials=N_TRIALS,
    seed=SEED,
)
tuning_result = tuner.run()
print(tuning_result.summary())
print(
    f"Tuning compute: {tuning_result.total_compute_time_sec:.2f}s"
    f" across {len(tuning_result.trials)} trial(s)."
)

if tuning_result.best is not None:
    print("\nRunning final test-set evaluation with best config …")
    runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
    final = runner.run(MLPMultivariateModel(), config=tuning_result.best.config)
    print(final.summary())
else:
    print("No successful trials — skipping final evaluation.")


### LamaH-CE Dynamic

In [ ]:
DATASET = "lamah-ce-dynamic"
base_cfg = make_config(DATASET)

tuner = HyperparameterTuner(
    model_factory=MLPMultivariateModel,
    base_config=base_cfg,
    dataset_key=DATASET,
    workspace_dir=WORKSPACE,
    search_space=mlp_multivariate_search_space(lr_low=LR_LOW, lr_high=LR_HIGH),
    strategy="random",
    n_trials=N_TRIALS,
    seed=SEED,
)
tuning_result = tuner.run()
print(tuning_result.summary())
print(
    f"Tuning compute: {tuning_result.total_compute_time_sec:.2f}s"
    f" across {len(tuning_result.trials)} trial(s)."
)

if tuning_result.best is not None:
    print("\nRunning final test-set evaluation with best config …")
    runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
    final = runner.run(MLPMultivariateModel(), config=tuning_result.best.config)
    print(final.summary())
else:
    print("No successful trials — skipping final evaluation.")


### NYC COVID

In [ ]:
DATASET = "nyc-covid"
base_cfg = make_config(DATASET)

tuner = HyperparameterTuner(
    model_factory=MLPMultivariateModel,
    base_config=base_cfg,
    dataset_key=DATASET,
    workspace_dir=WORKSPACE,
    search_space=mlp_multivariate_search_space(lr_low=LR_LOW, lr_high=LR_HIGH),
    strategy="random",
    n_trials=N_TRIALS,
    seed=SEED,
)
tuning_result = tuner.run()
print(tuning_result.summary())
print(
    f"Tuning compute: {tuning_result.total_compute_time_sec:.2f}s"
    f" across {len(tuning_result.trials)} trial(s)."
)

if tuning_result.best is not None:
    print("\nRunning final test-set evaluation with best config …")
    runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
    final = runner.run(MLPMultivariateModel(), config=tuning_result.best.config)
    print(final.summary())
else:
    print("No successful trials — skipping final evaluation.")


### Divvy Bikeshare

In [ ]:
DATASET = "divvy-bikeshare-static"
base_cfg = make_config(DATASET)

tuner = HyperparameterTuner(
    model_factory=MLPMultivariateModel,
    base_config=base_cfg,
    dataset_key=DATASET,
    workspace_dir=WORKSPACE,
    search_space=mlp_multivariate_search_space(lr_low=LR_LOW, lr_high=LR_HIGH),
    strategy="random",
    n_trials=N_TRIALS,
    seed=SEED,
)
tuning_result = tuner.run()
print(tuning_result.summary())
print(
    f"Tuning compute: {tuning_result.total_compute_time_sec:.2f}s"
    f" across {len(tuning_result.trials)} trial(s)."
)

if tuning_result.best is not None:
    print("\nRunning final test-set evaluation with best config …")
    runner = BenchmarkRunner(workspace_dir=WORKSPACE, datasets=[DATASET])
    final = runner.run(MLPMultivariateModel(), config=tuning_result.best.config)
    print(final.summary())
else:
    print("No successful trials — skipping final evaluation.")
